In [1]:
import os

# 注意：1- 代码放在整个文件的最上面；2- 0需要是字符串的类型
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import joblib
# from tensorflow.keras.preprocessing.text import Tokenizer   # 词汇映射器
from keras.src.legacy.preprocessing.text import Tokenizer

vocabs = ["周杰伦", "陈奕迅", "王力宏", "李宗盛", "吴亦凡", "鹿晗"]
# vocabs = ["过过", "过过", "过过"]   # 3个过过的one-hot是同一个

# 2- 训练词汇映射器
# 2.1- 创建词汇映射器
my_tokenizer = Tokenizer()
# 2.2- 用语料库训练
my_tokenizer.fit_on_texts(vocabs)
# 2.3- 得到训练好的词和索引的映射关系
# word_index类型是字典类型。key是词，value是词索引，索引从1开始
word_dict = my_tokenizer.word_index
# print(type(word_dict))
# print(word_dict)

# 3- 分别获得每个词的one-hot词向量
for word in vocabs:
    # 3.1- 初始化全0的列表
    one_hot = [0] * len(vocabs)
    # 3.2- 获得词在词汇映射器中的索引位置
    index = word_dict[word] - 1
    # 3.3- 将指定位置的值设置为1即可
    one_hot[index] = 1

    print(f"{word}-->{one_hot}")

# 4- 保存训练好的模型
joblib.dump(my_tokenizer, "../model/my_tokenizer.pkl")

周杰伦-->[1, 0, 0, 0, 0, 0]
陈奕迅-->[0, 1, 0, 0, 0, 0]
王力宏-->[0, 0, 1, 0, 0, 0]
李宗盛-->[0, 0, 0, 1, 0, 0]
吴亦凡-->[0, 0, 0, 0, 1, 0]
鹿晗-->[0, 0, 0, 0, 0, 1]


['../model/my_tokenizer.pkl']

In [6]:
from keras.src.legacy.preprocessing.text import Tokenizer

my_tokenizer = Tokenizer()
my_tokenizer.fit_on_texts(["张三是什么", "里斯"])
print(my_tokenizer.word_index)

{'张三是什么': 1, '里斯': 2}


In [11]:
vocabs = ["周杰伦", "陈奕迅", "周杰伦", "李宗盛", "吴亦凡", "鹿晗"]

# 2. 实例化Tokenizer并拟合语料库
# Tokenizer会自动构建词汇表，并将词语映射到从1开始的整数索引。
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts=vocabs)
print("词汇-索引 映射:", tokenizer.word_index)
print("索引-词汇 映射:", tokenizer.index_word)

# 3. 使用`texts_to_matrix`高效生成one-hot编码
# mode='binary'用于生成one-hot向量。
# Keras生成的矩阵第一列(索引0)是保留位，未使用。
# 通过切片[:, 1:]可以得到与原手动实现维度一致的结果。
one_hot_matrix = tokenizer.texts_to_matrix(vocabs, mode='binary')[:, 1:]

print('one_hot_matrix--->', one_hot_matrix)

print("--- One-Hot 编码 ---")
for word, vector in zip(vocabs, one_hot_matrix):
    # 将numpy浮点数向量转为整数列表，方便查看
    result = vector.astype(int).tolist()
print(f"{word} 的one-hot编码是: {result}")

# 4. 使用joblib保存Tokenizer对象，以便重用
tokenizer_path = './onehot_tokenizer.joblib'
joblib.dump(tokenizer, tokenizer_path)

# 5. 使用joblib加载Tokenizer对象，并生成"王力宏"的one-hot编码
print("--- One-Hot 编码 ---")
tokenizer = joblib.load(tokenizer_path)
result_ararry = tokenizer.texts_to_matrix(['王力宏'], mode='binary')[0, 1:]
result_list = result_ararry.astype(int).tolist()
print(f"'王力宏'的one-hot编码是: {result_list}")

词汇-索引 映射: {'周杰伦': 1, '陈奕迅': 2, '李宗盛': 3, '吴亦凡': 4, '鹿晗': 5}
索引-词汇 映射: {1: '周杰伦', 2: '陈奕迅', 3: '李宗盛', 4: '吴亦凡', 5: '鹿晗'}
one_hot_matrix---> [[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]
--- One-Hot 编码 ---
鹿晗 的one-hot编码是: [0, 0, 0, 0, 1]
--- One-Hot 编码 ---
'王力宏'的one-hot编码是: [0, 0, 0, 0, 0]


In [ ]:
def dm02_nnembeding_show():

        # 1 对句子分词 word_list
        sentence1 = '传智教育是一家上市公司，旗下有黑马程序员品牌。我是在黑马这里学习人工智能'
        sentence2 = "我爱自然语言处理"
        sentences = [sentence1, sentence2]
        word_list = []
        for s in sentences:
                word_list.append(jieba.lcut(s))
        # print('word_list--->', word_list)

        # 2 对句子word2id求my_token_list，对句子文本数值化sentence2id
        mytokenizer = Tokenizer()
        mytokenizer.fit_on_texts(texts=word_list)
        # print(mytokenizer.index_word, mytokenizer.word_index)

        # 打印my_token_list
        my_token_list = mytokenizer.index_word.values()
        print('my_token_list-->', my_token_list)

        # 打印文本数值化以后的句子
        sentence2id = mytokenizer.texts_to_sequences(texts=word_list)
        print('sentence2id--->', sentence2id, len(sentence2id))

        # 3 创建nn.Embedding层
        embd = nn.Embedding(num_embeddings=len(my_token_list), embedding_dim=8)
        # print("embd--->", embd)
        # print('nn.Embedding层词向量矩阵-->', embd.weight.data, embd.weight.data.shape, type(embd.weight.data))

        # 4 创建SummaryWriter对象 词向量矩阵embd.weight.data 和 词向量单词列表my_token_list
        # 词向量保存到runs目录中，不要出现中文路径，否则报错
        # log_dir: 默认None, 保存到当前目录下的runs/xxx目录(自动创建)中
        summarywriter = SummaryWriter(log_dir='D:/code/PycharmProjects/runs')
        # mat:词向量表示 张量或numpy数组
        # metadata:词标签
        summarywriter.add_embedding(mat=embd.weight.data, metadata=my_token_list)
        summarywriter.close()

        # 5 通过tensorboard观察词向量相似性
        # cd 程序的当前目录下执行下面的命令
        # 启动tensorboard服务 tensorboard --logdir=runs --host 0.0.0.0
        # 通过浏览器，查看词向量可视化效果 http://127.0.0.1:6006
        print('从nn.Embedding层中根据idx拿词向量')
        # # 6 从nn.Embedding层中根据idx拿词向量
        for idx in range(len(mytokenizer.index_word)):
                tmpvec = embd(torch.tensor(idx))
                print('%4s' % (mytokenizer.index_word[idx + 1]), tmpvec.detach().numpy())